# Week 74, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w074d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import ast
src = "def f(x):\n    if x > 0 and x < 10:\n        return [i for i in range(x) if i % 2]\n    return []\n"
func = ast.parse(src).body[0]
score = 1 + sum(isinstance(n, ast.If) + (len(n.values) - 1 if isinstance(n, ast.BoolOp) else 0)
                + (len(n.ifs) if isinstance(n, ast.comprehension) else 0) for n in ast.walk(func))
print(score)

**Predict before you run:** what will this print?



In [ ]:
import re
header = "@@ -10,3 +12 @@ def total():"
print(re.match(r"@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@", header).groups())

**Predict before you run:** what will this print?



In [ ]:
import doctest
parser = doctest.DocTestParser()
examples = parser.get_examples(">>> 1 + 1\n2\n>>> print('a')\na\n")
print(len(examples), [e.want for e in examples])

**Predict before you run:** what will this print?



In [ ]:
import fnmatch
print([fnmatch.fnmatchcase(b, "release/*") for b in ["release/1.0", "release/1.0/hotfix", "main"]])

**Predict before you run:** what will this print?



In [ ]:
msgs = ["feat: add search", "Fix typo", "fix(cart)!: change totals", "docs: readme"]
print([m.split(":")[0] for m in msgs if ":" in m])

### Exercise: Applying a patch

Write `apply_patch(text, diff)`: apply a unified diff for **one** file (with `---`, `+++` and one or more hunks) to `text`, returning the new text. Check each hunk's context and removed lines against the text at the hunk's old position, raising `ValueError("patch does not apply")` if they don't match. Lines are compared without their line endings, and the result ends with a newline if any lines remain.

In [ ]:
import re

def apply_patch(text, diff):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_patch():\n    \"context-checked patching\"\n    text = \"a\\nb\\nc\\nd\\ne\\n\"\n    diff = \"--- a/f\\n+++ b/f\\n@@ -2,2 +2,3 @@\\n b\\n-c\\n+C\\n+c2\\n@@ -5 +6,2 @@\\n e\\n+f\\n\"\n    assert apply_patch(text, diff) == \"a\\nb\\nC\\nc2\\nd\\ne\\nf\\n\"\n    try:\n        apply_patch(\"a\\nX\\nc\\nd\\ne\\n\", diff)\n        assert False\n    except ValueError as e:\n        assert str(e) == \"patch does not apply\"\n    assert apply_patch(\"x\\n\", \"--- a/f\\n+++ b/f\\n@@ -1 +0,0 @@\\n-x\\n\") == \"\"")

### Exercise: Counting examples

Write `examples_per_function(source)`: for each top-level function with a docstring, count its doctest examples with `doctest.DocTestParser().get_examples(docstring)`. Return a dict of name to count (functions without docstrings count 0), and `untested(source)`: the sorted names with no examples.

In [ ]:
import ast, doctest

def examples_per_function(source):
    ___

def untested(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_examples():\n    \"doctest examples per function\"\n    src = '''\ndef add(a, b):\n    \"\"\"Add.\n\n    >>> add(1, 2)\n    3\n    >>> add(-1, 1)\n    0\n    \"\"\"\n    return a + b\n\ndef neg(a):\n    \"\"\"Negate.\"\"\"\n    return -a\n\ndef bare(a):\n    return a\n'''\n    assert examples_per_function(src) == {\"add\": 2, \"neg\": 0, \"bare\": 0} and untested(src) == [\"bare\", \"neg\"]")

### Exercise: Forge: a review bot

Write `review_pr` and `summary` as described above.

In [ ]:
import ast, re

HUNK = re.compile(r"@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")
TYPES = ["feat", "fix", "docs", "refactor", "test", "chore", "perf", "build", "ci", "style", "revert"]


def _path(p):
    p = p.split("\t")[0].strip()
    return p[2:] if p.startswith(("a/", "b/")) else p


def parse_diff(text):
    files, current, hunk = [], None, None
    for line in text.splitlines():
        if line.startswith("--- "):
            current = {"old": _path(line[4:]), "new": None, "hunks": []}
            files.append(current)
            hunk = None
        elif line.startswith("+++ ") and current is not None and current["new"] is None:
            current["new"] = _path(line[4:])
        elif line.startswith("@@") and current is not None:
            a, b, c, d = HUNK.match(line).groups()
            hunk = {"old_start": int(a), "old_count": int(b) if b is not None else 1,
                    "new_start": int(c), "new_count": int(d) if d is not None else 1, "lines": []}
            current["hunks"].append(hunk)
        elif hunk is not None and line[:1] in (" ", "-", "+"):
            hunk["lines"].append((line[0], line[1:]))
    return files


def complexity(func):
    score = 1
    stack = list(ast.iter_child_nodes(func))
    while stack:
        node = stack.pop()
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.Lambda, ast.ClassDef)):
            continue
        if isinstance(node, (ast.If, ast.For, ast.AsyncFor, ast.While, ast.IfExp, ast.ExceptHandler, ast.match_case)):
            score += 1
        elif isinstance(node, ast.BoolOp):
            score += len(node.values) - 1
        elif isinstance(node, ast.comprehension):
            score += len(node.ifs)
        stack.extend(ast.iter_child_nodes(node))
    return score


def message_problems(message):
    lines = [l for l in message.splitlines() if not l.startswith("#")]
    while lines and not lines[-1].strip():
        lines.pop()
    if not lines:
        return ["empty message"]
    if lines[0].startswith("Merge "):
        return []
    problems = []
    if not re.fullmatch(r"(" + "|".join(TYPES) + r")(\([\w-]+\))?!?: \S.*", lines[0]):
        problems.append("bad first line")
    if len(lines[0]) > 72:
        problems.append("first line too long")
    if len(lines) > 1 and lines[1].strip():
        problems.append("no blank line after the first")
    return problems


def review_pr(diff, files, messages, hooks):
    ___


def summary(result):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("BIG = \"def route(x, y):\\n\" + \"\".join(f\"    if x == {i} or y == {i}:\\n        return {i}\\n\" for i in range(6)) + \"    return -1\\n\"\nFILES = {\n    \"src/shop.py\": \"def total(xs):\\n    print(xs)\\n    return sum(xs)\\n\\n\" + BIG,\n    \"tests/test_shop.py\": \"def test_total():\\n    assert True\\n\",\n    \"README.md\": \"# Shop  \\n\",\n}\nDIFF = (\n    \"--- a/src/shop.py\\n+++ b/src/shop.py\\n@@ -1,2 +1,3 @@\\n def total(xs):\\n+    print(xs)\\n     return sum(xs)\\n\"\n    \"@@ -4,0 +5,14 @@\\n\" + \"\".join(\"+\" + l + \"\\n\" for l in BIG.splitlines())\n    + \"--- a/README.md\\n+++ b/README.md\\n@@ -1 +1 @@\\n-# Shop\\n+# Shop  \\n\"\n)\n\ndef strip_ws(files):\n    return {n: \"\\n\".join(l.rstrip() for l in t.split(\"\\n\")) for n, t in files.items()}, False\n\ndef no_fail(files):\n    return files, False\n\nHOOKS = [{\"id\": \"trailing-whitespace\", \"run\": strip_ws}, {\"id\": \"toml-check\", \"run\": no_fail, \"files\": r\"\\.toml$\"}]\n\ndef test_review():\n    \"every check, combined\"\n    r = review_pr(DIFF, FILES, [\"feat(shop): routing\", \"wip\"], HOOKS)\n    assert r[\"stats\"] == (16, 1, 2) and r[\"size\"] == \"small\"\n    assert r[\"comments\"] == [\n        (\"*\", 0, \"code changed but no tests changed\"),\n        (\"src/shop.py\", 2, \"remove the debugging print\"),\n        (\"src/shop.py\", 5, \"route has complexity 13 (over 10)\"),\n    ], r[\"comments\"]\n    assert r[\"commit_problems\"] == [(2, \"bad first line\")]\n    assert r[\"hooks\"] == [(\"trailing-whitespace\", \"failed\"), (\"toml-check\", \"skipped\")]\n    assert r[\"verdict\"] == \"request changes\"\n\ndef test_clean():\n    \"a change that's fine\"\n    files = {\"src/a.py\": \"def f():\\n    return 1\\n\", \"tests/test_a.py\": \"def test_f():\\n    assert True\\n\"}\n    diff = (\"--- a/src/a.py\\n+++ b/src/a.py\\n@@ -1,2 +1,2 @@\\n def f():\\n-    return 0\\n+    return 1\\n\"\n            \"--- a/tests/test_a.py\\n+++ b/tests/test_a.py\\n@@ -1 +1,2 @@\\n def test_f():\\n+    assert True\\n\")\n    r = review_pr(diff, files, [\"fix: return one\"], HOOKS)\n    assert r == {\"stats\": (2, 1, 2), \"size\": \"small\", \"comments\": [], \"commit_problems\": [],\n                 \"hooks\": [(\"trailing-whitespace\", \"passed\"), (\"toml-check\", \"skipped\")], \"verdict\": \"approve\"}\n    assert summary(r) == (\"## Review summary\\n\\n**Size:** +2 -1 in 2 files (small)\\n\\n**Commits:** all good\\n\\n\"\n                          \"**Hooks:** trailing-whitespace passed, toml-check skipped\\n\\n**Comments:** none\\n\\n**Verdict:** approve\\n\")\n\ndef test_summary():\n    \"the comment's text\"\n    r = review_pr(DIFF, FILES, [\"feat(shop): routing\", \"wip\"], HOOKS)\n    text = summary(r)\n    assert \"**Commits:** commit 2: bad first line\" in text\n    assert \"- whole change: issue: code changed but no tests changed\" in text\n    assert \"- `src/shop.py:2`: issue: remove the debugging print\" in text\n    assert \"- `src/shop.py:5`: suggestion: route has complexity 13 (over 10)\" in text\n    assert text.endswith(\"**Verdict:** request changes\\n\")")

> **What a bot is for.** Look at the summary your bot writes for the first test: a debugging print, no tests, a bad commit message, a whitespace fix, and one complexity suggestion. A human reviewer would have spent their first ten minutes finding exactly those, and probably been less polite about the fourth. With the bot's comment at the top of the pull request, the author fixes them before anyone reads the change, and the reviewer starts at **purpose and design**, which only a person can judge. Note too what the bot marks as a suggestion rather than an issue: a machine can measure complexity, but whether it's acceptable is a judgement.

### Exercise: Forge bonus: a merge queue

A repository's **branch protection** for `main` says which checks are required, how many approving reviews are needed, whether a pull request must be **up to date** with `main`, and whether requested changes block merging. Write `merge_blockers(pr, rules)`: a list of reasons the pull request can't merge, in this order: `"draft"` if `pr["draft"]`; `"missing check: NAME"` for each required check not in `pr["checks"]`, and `"failing check: NAME"` for each required check whose result isn't `"success"`; `"needs N approvals (has M)"` if there are too few, counting each reviewer's **latest** review (a list of `(reviewer, state)` in time order, with states `"approved"`, `"changes_requested"` and `"commented"`, where a later `"commented"` doesn't replace an earlier decision); `"changes requested by NAME"` for each reviewer (sorted) whose latest decision is `"changes_requested"`, if `rules["block_on_changes"]`; `"out of date with main"` if `rules["up_to_date"]` and `pr["base"] != rules["main_head"]`. Then `merge_queue(prs, rules)`: the list of `(number, blockers)` in number order, and the numbers that can merge now.

In [ ]:
def merge_blockers(pr, rules):
    ___

def merge_queue(prs, rules):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("RULES = {\"required\": [\"lint\", \"test (3.12)\", \"test (3.13)\"], \"approvals\": 2, \"block_on_changes\": True, \"up_to_date\": True, \"main_head\": \"abc\"}\n\ndef test_blockers():\n    \"branch protection rules\"\n    ready = {\"number\": 1, \"draft\": False, \"base\": \"abc\", \"checks\": {\"lint\": \"success\", \"test (3.12)\": \"success\", \"test (3.13)\": \"success\", \"docs\": \"failure\"},\n             \"reviews\": [(\"ada\", \"approved\"), (\"bob\", \"commented\"), (\"bob\", \"approved\")]}\n    assert merge_blockers(ready, RULES) == []\n    messy = {\"number\": 2, \"draft\": True, \"base\": \"old\", \"checks\": {\"lint\": \"success\", \"test (3.12)\": \"failure\"},\n             \"reviews\": [(\"ada\", \"approved\"), (\"cy\", \"changes_requested\"), (\"ada\", \"changes_requested\"), (\"cy\", \"commented\")]}\n    assert merge_blockers(messy, RULES) == [\"draft\", \"missing check: test (3.13)\", \"failing check: test (3.12)\", \"needs 2 approvals (has 0)\",\n                                            \"changes requested by ada\", \"changes requested by cy\", \"out of date with main\"]\n    changed_mind = dict(ready, reviews=[(\"ada\", \"changes_requested\"), (\"ada\", \"approved\"), (\"bob\", \"approved\")])\n    assert merge_blockers(changed_mind, RULES) == []\n    lenient = dict(RULES, block_on_changes=False, up_to_date=False, approvals=1)\n    assert merge_blockers(dict(messy, draft=False, checks=ready[\"checks\"]), lenient) == [\"needs 1 approvals (has 0)\"]\n\ndef test_queue():\n    \"what can merge now\"\n    prs = [\n        {\"number\": 7, \"draft\": False, \"base\": \"abc\", \"checks\": {\"lint\": \"success\", \"test (3.12)\": \"success\", \"test (3.13)\": \"success\"}, \"reviews\": [(\"a\", \"approved\"), (\"b\", \"approved\")]},\n        {\"number\": 3, \"draft\": False, \"base\": \"abc\", \"checks\": {}, \"reviews\": []},\n    ]\n    report, ready = merge_queue(prs, RULES)\n    assert ready == [7] and report[0][0] == 3 and report[1] == (7, [])")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Forge: a review bot

Write **`review_pr(diff, files, messages, hooks)`**, a bot's first pass over a pull request, and **`summary(result)`**, the comment it posts. `diff` is the pull request's unified diff; `files` maps each path to its **new** contents; `messages` is the list of commit messages; `hooks` is a list of dicts with `"id"`, `"run"` (a function from a dict of files to `(new_files, failed)`, as on Day 5) and an optional `"files"` regular expression. `parse_diff`, `complexity` (of one function node, its own body only) and `message_problems` (Day 5's commit message checks, returning a list) are given.

`review_pr` returns a dict with:

- `"stats"`: `(added, removed, files)`: the counts of `+` and `-` lines in hunks, and the number of files in the diff;
- `"size"`: `"small"` up to 100 changed lines, `"fine"` up to 400, otherwise `"split it: N lines"`;
- `"comments"`: a sorted list of `(path, line, message)`, with the checks from Day 2's challenge on added lines of `.py` files (`"remove the debugging print"` unless under `scripts/`, `"bare except hides errors"`, `"TODO without an issue number"`, `"line too long (N > 100)"`, and `("*", 0, "code changed but no tests changed")`), plus, for each function in a changed `.py` file's new contents that contains an added line and has complexity over 10, `(path, def_line, "NAME has complexity N (over 10)")`;
- `"commit_problems"`: a list of `(number, problem)`, numbering the commits from 1;
- `"hooks"`: a list of `(id, result)`: each hook runs on the files from `files` that appear in the diff and match its `"files"` pattern (all of them if there's none); `"skipped"` with no files, `"failed"` if it failed or changed anything, otherwise `"passed"`;
- `"verdict"`: `"request changes"` if there's any **blocking** finding (a comment about a debugging print, a bare except or missing tests; any commit problem; any failed hook), otherwise `"approve"`.

`summary(result)` writes the comment, with blank lines between its parts: `## Review summary`; `**Size:** +A -R in N files (SIZE)` (`file` for one); `**Commits:** all good` or the problems as `commit N: PROBLEM` joined with `"; "`; `**Hooks:** id result, id result` (or `none`); then `**Comments:** none`, or `**Comments:**` followed by one line per comment, `` - `path:line`: LABEL: message `` (or `- whole change: LABEL: message` for path `*`), where LABEL is `issue` for blocking comments and `suggestion` for the rest; and last `**Verdict:** VERDICT`, ending with a newline.

## Forge bonus: a merge queue

## Weekly quiz

## Week 74 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can measure complexity and nesting, write the Python idioms, configure a linter, and fix code safely with tokens. ([Day 1](w074d1.html))
- I can read and parse unified diffs, review in order, and write labelled, useful comments. ([Day 2](w074d2.html))
- I can write PEP 257 docstrings and doctests, generate reference docs, and check docs against the code. ([Day 3](w074d3.html))
- I can read a CI workflow, expand a matrix, order jobs, and keep CI fast, safe and reproducible. ([Day 4](w074d4.html))
- I can configure pre-commit, write checking and fixing hooks, and check commit messages. ([Day 5](w074d5.html))
- I built a review bot and a merge queue. (Today)

Tomorrow is a **rest day**. Next week is **databases**: SQL from `SELECT` to joins and aggregation, Python's `sqlite3`, object-relational mappers, and transactions and indexes.